# 🤖 Gemini + Tools: How Does an LLM Decide When to Use a Tool?

### Dinesh AI Academy — Day 3: Tools & Workflows

In this notebook we will build a **real function-calling loop** using the Google GenAI Python SDK.

The key idea:

> **The LLM does not execute our Python function. The LLM requests a tool call, our application executes the function, and the result is sent back to the LLM.**

We will build two simple tools:

- 🧮 `calculate()` — performs arithmetic
- 🕐 `get_current_time()` — returns the current time for a requested timezone

We will deliberately use **manual function calling** in the main demo so students can see the complete loop:

```text
User
  ↓
Gemini
  ↓
Does a tool help?
  ↓
Tool call + arguments
  ↓
Our Python application
  ↓
Execute the real function
  ↓
Tool result
  ↓
Gemini
  ↓
Final answer
```

> **Teaching goal:** Do not focus only on the code. Focus on the decision flow.

## 1. What is a Tool?

A **tool** is an external capability that an LLM can request your application to use.

Examples:

| Tool | Capability |
|---|---|
| Calculator | Perform reliable calculations |
| Weather API | Get current weather |
| Database | Retrieve business data |
| Gmail API | Read/send email |
| Calendar API | Read/create meetings |
| Search | Find current information |
| Python | Execute code |
| CRM API | Retrieve/update customer data |

A useful mental model:

**LLM = decision/reasoning interface**  
**Tool = capability/action**  
**Your application = executor/orchestrator**

Important: the model does not directly run your Python function. Your application does.

In [ ]:
# Install the current Google GenAI Python SDK.
# In Google Colab, run this cell once.

!pip -q install -U google-genai

## 2. Add Your Gemini API Key

Create a Gemini API key in Google AI Studio.

For a classroom demo, the easiest approach in Colab is to enter the key when prompted.

**Never publish your API key in a notebook, GitHub repository, Moodle, WhatsApp group, or screenshot.**

In [ ]:
import os
from getpass import getpass

if not os.environ.get("GEMINI_API_KEY"):
    os.environ["GEMINI_API_KEY"] = getpass("Enter your GEMINI_API_KEY: ")

from google import genai
from google.genai import types

client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])

# You can change this model if your account has access to another Gemini model.
MODEL = "gemini-2.5-flash"

print("Gemini client is ready.")
print("Model:", MODEL)

## 3. First: Create Normal Python Functions

Before we call them "AI tools", these are simply normal Python functions.

This is an important teaching point:

> **A tool is usually just application code that we expose to the model through a tool/function declaration.**

In [ ]:
from datetime import datetime
from zoneinfo import ZoneInfo

def calculate(a: float, b: float, operation: str) -> float:
    """Perform a basic arithmetic calculation."""
    if operation == "add":
        return a + b
    elif operation == "subtract":
        return a - b
    elif operation == "multiply":
        return a * b
    elif operation == "divide":
        if b == 0:
            raise ValueError("Cannot divide by zero.")
        return a / b
    else:
        raise ValueError(f"Unsupported operation: {operation}")


def get_current_time(timezone: str) -> dict:
    """Get the current date and time for an IANA timezone such as Asia/Tokyo."""
    now = datetime.now(ZoneInfo(timezone))
    return {
        "timezone": timezone,
        "date": now.strftime("%Y-%m-%d"),
        "time": now.strftime("%H:%M:%S"),
        "formatted": now.strftime("%A, %d %B %Y at %I:%M:%S %p")
    }


# Test the functions directly.
print("Calculator:", calculate(25, 40, "multiply"))
print("Current time:", get_current_time("Asia/Kolkata"))

## 4. Give Gemini a Description of the Tools

Gemini cannot guess the interface of your Python functions.

We describe:

1. Tool name
2. What it does
3. Input parameters
4. Required parameters

Think of this as giving the model a **menu of capabilities**.

The model can then decide whether a tool is relevant to the user's request.

In [ ]:
calculator_declaration = {
    "name": "calculate",
    "description": "Performs basic arithmetic calculations.",
    "parameters": {
        "type": "object",
        "properties": {
            "a": {
                "type": "number",
                "description": "The first number."
            },
            "b": {
                "type": "number",
                "description": "The second number."
            },
            "operation": {
                "type": "string",
                "description": "The arithmetic operation.",
                "enum": ["add", "subtract", "multiply", "divide"]
            }
        },
        "required": ["a", "b", "operation"]
    }
}

time_declaration = {
    "name": "get_current_time",
    "description": "Gets the current date and time for an IANA timezone such as Asia/Kolkata or Asia/Tokyo.",
    "parameters": {
        "type": "object",
        "properties": {
            "timezone": {
                "type": "string",
                "description": "IANA timezone name, for example Asia/Kolkata, Asia/Tokyo, or America/New_York."
            }
        },
        "required": ["timezone"]
    }
}

tools = types.Tool(
    function_declarations=[
        calculator_declaration,
        time_declaration
    ]
)

config = types.GenerateContentConfig(
    tools=[tools],
    system_instruction=(
        "You are a helpful AI assistant. "
        "Use the available tools when they are useful or required. "
        "For current time questions, use get_current_time instead of guessing. "
        "For calculations where accuracy matters, use calculate."
    )
)

print("Two tools are available to Gemini.")

## 5. Ask Gemini a Question and Inspect Its Decision

We intentionally **do not execute the function automatically**.

This lets us see what Gemini requested.

Try:

> `What time is it in Tokyo right now?`

The expected flow is:

```text
User question
     ↓
Gemini
     ↓
function_call: get_current_time
     ↓
arguments: {"timezone": "Asia/Tokyo"}
```

Notice the important point:

**Gemini returns a request. It does not run our Python function.**

In [ ]:
user_prompt = "What time is it in Tokyo right now?"

response = client.models.generate_content(
    model=MODEL,
    contents=user_prompt,
    config=config
)

print("USER:")
print(user_prompt)

print("\nGEMINI RESPONSE PARTS:")
for i, part in enumerate(response.candidates[0].content.parts):
    print(f"Part {i}:")
    print("  text:", part.text)
    print("  function_call:", part.function_call)

## 6. Extract the Tool Call

A function call contains:

- **name** → which tool Gemini wants
- **args** → the arguments Gemini wants your application to use
- **id** → an identifier used to associate the tool result with the request

Conceptually:

```json
{
  "name": "get_current_time",
  "args": {
    "timezone": "Asia/Tokyo"
  }
}
```

In [ ]:
function_call = None

for part in response.candidates[0].content.parts:
    if part.function_call:
        function_call = part.function_call
        break

if function_call:
    print("🔧 TOOL REQUESTED")
    print("Tool:", function_call.name)
    print("Arguments:", dict(function_call.args))
    print("Call ID:", function_call.id)
else:
    print("No tool call was requested.")
    print("Gemini answered directly:")
    print(response.text)

## 7. Our Application Executes the Tool

This is the most important line in the architecture:

```text
Gemini → "Please call get_current_time(...)"
                    ↓
             OUR PYTHON CODE
                    ↓
             get_current_time(...)
```

The **application**, not the LLM, executes the function.

In [ ]:
def execute_tool(name, args):
    """Execute the tool requested by Gemini."""

    if name == "calculate":
        return calculate(
            a=float(args["a"]),
            b=float(args["b"]),
            operation=args["operation"]
        )

    if name == "get_current_time":
        return get_current_time(
            timezone=args["timezone"]
        )

    raise ValueError(f"Unknown tool requested: {name}")


if function_call:
    tool_result = execute_tool(
        function_call.name,
        dict(function_call.args)
    )

    print("⚙️ TOOL EXECUTED")
    print("Tool:", function_call.name)
    print("Result:", tool_result)

## 8. Send the Tool Result Back to Gemini

Now we complete the loop.

We send Gemini:

1. The original user request
2. Gemini's previous tool-call message
3. The result produced by our application

Gemini can now turn the raw tool result into a natural-language answer.

In [ ]:
if function_call:
    # Preserve Gemini's tool-call response in the conversation.
    contents = [
        types.Content(
            role="user",
            parts=[types.Part.from_text(text=user_prompt)]
        ),
        response.candidates[0].content
    ]

    # Add the function result.
    function_response_part = types.Part.from_function_response(
        name=function_call.name,
        response=tool_result,
        id=function_call.id
    )

    contents.append(
        types.Content(
            role="user",
            parts=[function_response_part]
        )
    )

    final_response = client.models.generate_content(
        model=MODEL,
        contents=contents,
        config=config
    )

    print("🤖 FINAL GEMINI ANSWER:")
    print(final_response.text)

# 🎯 9. See the Complete Tool-Calling Loop

You have now implemented:

```text
┌─────────────────────────┐
│          USER           │
│ "What time is Tokyo?"   │
└────────────┬────────────┘
             ↓
┌─────────────────────────┐
│         GEMINI          │
│                         │
│ Decide: tool required?  │
└────────────┬────────────┘
             ↓
       YES → Tool Call
             ↓
┌─────────────────────────┐
│     YOUR APPLICATION    │
│                         │
│ execute_tool(...)       │
└────────────┬────────────┘
             ↓
┌─────────────────────────┐
│      REAL PYTHON TOOL   │
│                         │
│ get_current_time(...)   │
└────────────┬────────────┘
             ↓
          Result
             ↓
┌─────────────────────────┐
│         GEMINI          │
│                         │
│ Create final response   │
└────────────┬────────────┘
             ↓
           USER
```

### The key distinction

**LLM decides → Application executes → LLM responds**

## 10. A Query That Should NOT Need a Tool

Now ask something that is purely conversational.

Try:

> `Explain RAG in simple words.`

Gemini should normally answer directly because neither calculator nor current-time data is required.

This demonstrates that **tool use is conditional**.

```text
Question
   ↓
Gemini
   ↓
Does a tool help?
   ├── No  → Answer directly
   └── Yes → Request tool
```

In [ ]:
def ask_without_forcing_tools(prompt):
    response = client.models.generate_content(
        model=MODEL,
        contents=prompt,
        config=config
    )

    tool_calls = [
        part.function_call
        for part in response.candidates[0].content.parts
        if part.function_call
    ]

    print("USER:", prompt)
    print("TOOL CALLS:", len(tool_calls))
    print("ANSWER:", response.text if response.text else "(No direct text; model requested a tool.)")

ask_without_forcing_tools("Explain RAG in simple words.")

## 11. Try a Calculation

Now ask:

> `Calculate 1250 × 48 and tell me the answer.`

Depending on the model's decision and configuration, Gemini may request the calculator tool or may answer directly.

For teaching, this is useful: **having a tool available does not mean the model must always use it.**

If you want to force a tool for a controlled classroom demonstration, use tool-calling configuration rather than relying on the model's preference.

In [ ]:
user_prompt = "Calculate 1250 multiplied by 48. Accuracy is important."

response = client.models.generate_content(
    model=MODEL,
    contents=user_prompt,
    config=config
)

for part in response.candidates[0].content.parts:
    if part.function_call:
        print("🔧 Gemini requested:", part.function_call.name)
        print("Arguments:", dict(part.function_call.args))
    elif part.text:
        print("💬 Gemini text:", part.text)

# 🧪 12. Classroom Challenge

Ask students to predict whether Gemini will use a tool.

| User request | Likely useful tool |
|---|---|
| What time is it in London? | `get_current_time` |
| What is 25% of 800? | `calculate` |
| Explain what an LLM is | None |
| What time is it in New York? | `get_current_time` |
| Multiply 1299 by 37 | `calculate` |
| What is RAG? | None |

The important lesson is not the exact prediction.

The important lesson is:

> **The model receives a list of capabilities and can request a capability when the user's goal requires it.**

# 🔥 13. Upgrade the Demo: Two Tools + One User Request

Try a request that can require multiple steps:

> **"What time is it in Tokyo, and calculate 15% of 8500."**

Gemini may request one or both tools. Depending on the model/configuration, calls can be sequential or parallel.

This is the bridge to **workflows and agents**.

```text
                    User
                      ↓
                    Gemini
                 ↙          ↘
       Time Tool              Calculator
           ↓                      ↓
      Tokyo time                1275
                 ↘          ↙
                    Gemini
                      ↓
                 Final answer
```

In [ ]:
user_prompt = (
    "What time is it in Tokyo right now, and calculate 15 percent of 8500. "
    "Use tools where appropriate."
)

response = client.models.generate_content(
    model=MODEL,
    contents=user_prompt,
    config=config
)

print("USER:")
print(user_prompt)

print("\nMODEL TOOL REQUESTS:")
found = False

for part in response.candidates[0].content.parts:
    if part.function_call:
        found = True
        fc = part.function_call
        print(f"- {fc.name} -> {dict(fc.args)}")

if not found:
    print("No function call was requested in this response.")

# 🧠 14. Tools vs Workflows vs Agents

This is the most important conceptual takeaway for Day 3 and Day 4.

### Tool

> **A capability**

```text
Calculator
Weather API
Database
Email
Search
```

### Workflow

> **A predefined sequence of steps**

```text
Input
 ↓
Classify
 ↓
Retrieve
 ↓
Generate
 ↓
Validate
 ↓
Output
```

### Agent

> **A system where the model can decide what action/tool/step to take next toward a goal.**

```text
Goal
 ↓
Agent
 ↓
Decide
 ↓
Tool
 ↓
Observe
 ↓
Decide again
 ↓
Tool
 ↓
Final result
```

### One sentence for students

> **Tool = capability. Workflow = process. Agent = decision-making over capabilities and steps.**

# 🛡️ 15. Production Safety Note

Never expose unrestricted tools to an LLM.

For real applications:

- Validate tool arguments
- Allow-list tools
- Check user permissions
- Protect secrets and API keys
- Add authentication/authorization
- Add timeouts
- Handle tool failures
- Log tool calls
- Add human approval before high-impact actions
- Never let model-generated arguments bypass your application's security controls

For example:

```text
LLM requests:
send_email(to="someone@example.com")

        ↓

Your application:
Is this user allowed?
Is the recipient valid?
Does policy permit this action?
Does this require human approval?

        ↓

Execute / Reject
```

The LLM can **request** an action. Your application should remain in control of whether that action is actually executed.

# 🎓 Day 3 Takeaway

By the end of this notebook, students should be able to explain:

1. What a tool is
2. Why an LLM needs tool declarations
3. How the model chooses/request a tool
4. What a function call contains
5. How the application executes the function
6. How the result goes back to the LLM
7. How the LLM creates the final response
8. The difference between tools, workflows, and agents

### The complete mental model

```text
Natural Language
      ↓
     LLM
      ↓
Tool decision
      ↓
Function call
      ↓
Your application
      ↓
Real-world tool/API/code
      ↓
Tool result
      ↓
     LLM
      ↓
Final answer
```

## 🚀 Next: Day 4 — AI Agents & MCP

Tomorrow we take the next step:

**What happens when the AI can choose tools, decide the next step, observe results, and continue until the goal is achieved?**

That is where **AI Agents** begin.

## Official references

- Gemini API — Function calling: https://ai.google.dev/gemini-api/docs/function-calling
- Gemini API — Getting started: https://ai.google.dev/gemini-api/docs/get-started
- Gemini API — Tools: https://ai.google.dev/gemini-api/docs/tools
- Google AI Studio: https://aistudio.google.com/

This notebook intentionally uses the lower-level/manual function-calling flow for teaching visibility. The current Google GenAI Python SDK also supports automatic function calling, where the SDK can execute Python functions for the model; manual calling is better for demonstrating the architecture.